# transformer_ko4 — knockout-context transformer with spatial gating (Colab / GPU)

This runs **v4** of the knockout far-field transformer: a knockout is a *set of tokens* (the perturbed gene + its post-KO
co-dependency / complex / PPI partners), each token carrying `[DepMap profile | network-SVD | essentiality | role]`,
**physics edge-features** (interface n_pdb, co-dependency magnitude, shared-complex count), a **Graphormer edge-conditioned
attention** (learned bias on edge-type + magnitude between every token pair), and now **spatial gating** (GO compartment: a
same-compartment attention bias / hard mask). It is trained as a *learned-retrieval metric* and scored on held-out K562
Perturb-seq, tide-removed specific-mover recall@50, against a tide-null floor and the retrieval oracle.

**Honest expectation:** the gap to the ~0.62 oracle is data/readout-limited (a same-KO cross-line probe showed the specific
far-field is ~85% context-specific), so spatial gating is expected to be *marginal* on this 96h steady-state mRNA readout.
This notebook is the honest test — and a GPU base you can extend toward the GPU-gated layers (AF-Multimer, ΔΔG).

### Prerequisites (one-time)
You need three data artifacts in your Google Drive (the project already persists them via `persist.py`):
`cell_complete.json`, `depmap_vecs.npz`, `nlz_K562.pkl`. Cell 3 searches common Drive locations and copies them in.
Set **Runtime → Change runtime type → GPU** for speed.


In [ ]:
# 1. deps (Colab already has torch+CUDA, numpy, scipy)
import torch, sys
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!pip -q install scipy >/dev/null 2>&1


In [ ]:
# 2. clone the repo (brings colab/*.py + the tracked data: trrust_regulon.json, iface_gate.json)
import os
BRANCH = 'claude/vectorize-gex-propensity-zp09w8'
if not os.path.isdir('/content/cell'):
    !git clone --branch $BRANCH --depth 1 https://github.com/Nikku03/cell.git /content/cell
%cd /content/cell
!ls colab/transformer_ko4.py && echo OK


In [ ]:
# 3. mount Drive and stage the 3 gitignored artifacts into the paths the code expects
from google.colab import drive
import glob, shutil, os
drive.mount('/content/drive')
SP = '/tmp/claude-0/-home-user-cell/0f039315-b3a9-52ac-8187-9fae0d726994/scratchpad'   # eval_harness looks here for nlz_K562.pkl
os.makedirs(SP, exist_ok=True); os.makedirs('/content/cell/outputs/orphan', exist_ok=True)
NEED = {'cell_complete.json':'/content/cell/outputs/orphan/cell_complete.json',
        'depmap_vecs.npz':'/content/cell/outputs/orphan/depmap_vecs.npz',
        'nlz_K562.pkl': SP + '/nlz_K562.pkl'}
SEARCH = ['/content/drive/MyDrive/cell_model/artifacts','/content/drive/MyDrive/cell_model/caches',
          '/content/drive/MyDrive/nexus_cache','/content/drive/MyDrive/cell_model','/content/drive/MyDrive']
def find(name):
    for d in SEARCH:
        hits = glob.glob(os.path.join(d, '**', name), recursive=True)
        if hits: return hits[0]
    return None
missing = []
for name, dst in NEED.items():
    if os.path.exists(dst): print('present:', name); continue
    src = find(name)
    if src: shutil.copy(src, dst); print(f'copied {name}  <-  {src}')
    else: missing.append(name)
assert not missing, ('MISSING in Drive: ' + ', '.join(missing) +
    '  -> put them under MyDrive/cell_model/ (artifacts or caches) and re-run this cell.')
print('all artifacts staged.')


In [ ]:
# 4. (optional) fast end-to-end smoke check on GPU: 1 seed, reduced configs/epochs (~1-2 min)
!cd /content/cell && V4_SMOKE=1 python colab/transformer_ko4.py


In [ ]:
# 5. FULL run: 3 splits x {v3-full base, +spatial-bias, +spatial-mask} + controls, GPU
!cd /content/cell && python colab/transformer_ko4.py


### Reading the output
- **`v3-full (no spatial)`** is the base (edge-features + Graphormer edge-bias, no spatial).
- **`+spatial-bias` / `+spatial-mask`** add the GO-compartment attention bias / hard co-localization gate.
- **`spatial - base`** is the number that matters; **`TIDE-null`** (~0.25) is the floor, **`ORACLE*`** (~0.61) the retrieval ceiling,
  and **`wrong-KO shuffle`** is the identity control (should collapse).
- If `spatial - base` is ~0 (the honest prior), localization was already latent in the network-SVD and the ceiling is data-limited,
  not localization-limited. If it is a stable positive with the shuffle control passing, spatial gating genuinely helps.

To push past the retrieval ceiling you would need **transient (2–12h) kinetics** data (not steady-state) — the architecture here is
ready for it; swap the target matrix and re-run.
